# 02 - Analyse du risque de taux et de spread

À lancer **souvent** : tout est relu depuis les caches créés par `01_collecte.ipynb`, en quelques secondes.

1. Obligations encore vivantes
2. Prix, durations, convexité, key rate durations (une ligne par tranche)
3. Synthèse par secteur
4. Stress tests : choc parallèle, scénarios de déformation de courbe, choc de spread
5. Exports : `data/resultats.csv` (pour l'application), `data/rapport_resultats.xlsx`, `data/rapport.html`

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

from courbe import get_courbe_bam
from pipeline import (charger_tranches, obligations_vivantes, analyser_marche, resume_secteurs,
                      impact_scenario, tous_les_scenarios, FICHIER_RESULTATS)
from calculs import SCENARIOS
from rapport import exporter_excel, exporter_html

courbe = get_courbe_bam()          # relit data/courbe_bam.csv
tranches = charger_tranches()      # relit data/tranches.csv
print("Courbe du", pd.to_datetime(courbe["date_valeur"]).iloc[0].date(), "|", len(tranches), "tranches")

Courbe du 2026-11-12 | 472 tranches


## 1. Obligations encore vivantes

In [2]:
vivantes = obligations_vivantes(tranches)
print(len(vivantes), "tranches vivantes, complètes et sans doublon")
print(vivantes["nature_taux"].value_counts().to_string())
print("Échéances estimées (visa + maturité) :", int(vivantes["echeance_estimee"].sum()))

162 tranches vivantes, complètes et sans doublon
nature_taux
révisable    119
fixe          43
Échéances estimées (visa + maturité) : 43


## 2. Analyse de chaque tranche

Flux datés en fractions d'année, chacun actualisé à **courbe BAM(t) + prime**. Colonnes principales :
`prix_plein`, `coupon_couru`, `prix_pied`, `macaulay`, `modifiee` (duration de taux effective), `convexite`,
`duration_spread`, `dv01` (MAD par titre de 100 000), `seuil_pb`, chocs parallèles et `KRD …` (key rate durations).

In [3]:
resultats = analyser_marche(vivantes, courbe)
resultats.to_csv(FICHIER_RESULTATS, index=False)       # lu par l'application Streamlit
resultats.head(10)

,emetteur,tranche,type,secteur,nature_taux,remboursement,echeance,differe_ans,maturite_ans,echeance_estimee,maturite_res,prochain_flux_ans,taux_bam_%,prime_pb,coupon_%,ytm_%,prix_plein,coupon_couru,prix_pied,macaulay,modifiee,convexite,duration_spread,dv01,seuil_pb,choc_-100pb_%,choc_+50pb_%,choc_+100pb_%,choc_+200pb_%,KRD 0.25a,KRD 1a,KRD 2a,KRD 5a,KRD 10a,KRD 15a,KRD 30a
0,BMCE BANK,B,subordonné,Banque,révisable,in fine,2027-11-21,0,10.0,True,1.02,0.02,2.37,90.0,3.263,3.263,103.186,3.182,100.003,0.025,0.024,0.02,0.962,0.25,NaN,0.02,-0.01,-0.02,-0.05,0.024,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
1,CIH BANK,D,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,75.0,3.113,3.113,102.983,2.976,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2,CIH BANK,F,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,95.0,3.313,3.313,103.173,3.167,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
3,CIH BANK,E,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,80.0,3.163,3.163,103.030,3.024,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
4,CIH BANK,B,subordonné,Banque,fixe,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,85.0,3.020,3.020,102.886,2.888,99.998,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
5,CIH BANK,C,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,60.0,2.963,2.963,102.839,2.833,100.007,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
6,CIH BANK,A,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,70.0,3.063,3.063,102.935,2.928,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
7,CIH BANK,G,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,70.0,3.063,3.063,102.935,2.928,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
8,CIH BANK,H,subordonné,Banque,révisable,in fine,2026-11-28,0,10.0,True,0.04,0.04,2.17,85.0,3.213,3.213,103.078,3.072,100.006,0.044,0.043,0.04,0.043,0.44,NaN,0.04,-0.02,-0.04,-0.08,0.043,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
9,CRÉDIT DU MAROC,D,subordonné,Banque,révisable,in fine,2026-11-30,0,10.0,True,0.05,0.05,2.17,95.0,3.313,3.313,103.156,3.149,100.007,0.049,0.048,0.05,0.048,0.49,NaN,0.05,-0.02,-0.05,-0.09,0.048,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0


## 3. Synthèse par secteur

In [4]:
resume_secteurs(resultats)

,tranches,part_revisable_pct,prime_moy_pb,ytm_moy_pct,duration_taux_moy,duration_spread_moy,seuil_median_pb
secteur,,,,,,,
Société de financement,10,100,71.50,3.08,0.21,3.81,492.0
Immobilier,8,50,248.75,5.00,0.88,1.65,464.0
Banque,88,80,68.86,3.16,0.95,2.68,355.0
Industrie & services,13,54,110.00,3.55,1.18,1.80,361.0
Établissement public,43,65,89.13,3.43,1.38,4.33,336.0


In [5]:
pd.crosstab(resultats["secteur"], resultats["nature_taux"], margins=True, margins_name="Total")

nature_taux,fixe,révisable,Total
secteur,,,
Banque,18,70,88
Immobilier,4,4,8
Industrie & services,6,7,13
Société de financement,0,10,10
Établissement public,15,28,43
Total,43,119,162


### Tranches à taux fixe : les plus résistantes et les plus exposées

In [6]:
fixes = resultats[resultats["nature_taux"] == "fixe"]
colonnes = ["emetteur", "tranche", "secteur", "maturite_res", "ytm_%", "modifiee", "convexite", "seuil_pb", "choc_+100pb_%"]
print("Plus résistantes :"); display(fixes.nsmallest(5, "modifiee")[colonnes])
print("Plus exposées :");    display(fixes.nlargest(5, "modifiee")[colonnes])

Plus résistantes :


,emetteur,tranche,secteur,maturite_res,ytm_%,modifiee,convexite,seuil_pb,choc_+100pb_%
4,CIH BANK,B,Banque,0.04,3.02,0.043,0.04,NaN,-0.04
12,CRÉDIT DU MAROC,B,Banque,0.05,3.12,0.048,0.05,NaN,-0.05
23,MANAGEM,B,Industrie & services,0.11,3.27,0.103,0.11,NaN,-0.10
49,FEC,D,Établissement public,0.13,3.17,0.125,0.14,NaN,-0.12
50,FEC,C,Établissement public,0.13,3.17,0.125,0.14,NaN,-0.12


Plus exposées :


,emetteur,tranche,secteur,maturite_res,ytm_%,modifiee,convexite,seuil_pb,choc_+100pb_%
161,CDM,A,Banque,8.96,4.352,7.267,65.39,60.0,-6.95
160,CFG BANK,A,Banque,9.13,4.300,7.148,65.42,60.0,-6.83
158,ONCF,B,Établissement public,8.66,5.037,6.758,58.15,75.0,-6.48
159,ONCF,A,Établissement public,8.66,5.037,6.758,58.15,75.0,-6.48
157,FEC,B,Établissement public,6.00,4.051,5.227,33.98,77.0,-5.06


## 4. Scénarios de déformation de la courbe

Chocs en points de base aux points de la courbe, interpolés entre les points (inspirés des scénarios IRRBB de Bâle).

In [7]:
pd.DataFrame(SCENARIOS, index=["3 mois", "1 an", "2 ans", "5 ans", "10 ans", "15 ans", "30 ans"]).T

,3 mois,1 an,2 ans,5 ans,10 ans,15 ans,30 ans
Hausse parallèle,100,100,100,100,100,100,100
Baisse parallèle,-100,-100,-100,-100,-100,-100,-100
Pentification,-65,-50,-35,0,50,75,90
Aplatissement,90,75,55,20,-20,-35,-45
Hausse des taux courts,150,120,90,45,15,5,0
Baisse des taux courts,-150,-120,-90,-45,-15,-5,0
"BAM +50 pb, 10 ans -20 pb",50,50,35,10,-20,-20,-20


In [8]:
scenarios = tous_les_scenarios(resultats, courbe)
scenarios.groupby("secteur")[list(SCENARIOS)].mean().round(2)

,Hausse parallèle,Baisse parallèle,Pentification,Aplatissement,Hausse des taux courts,Baisse des taux courts,"BAM +50 pb, 10 ans -20 pb"
secteur,,,,,,,
Banque,-0.93,0.98,0.13,-0.34,-0.68,0.70,-0.19
Immobilier,-0.87,0.90,0.33,-0.52,-0.87,0.89,-0.32
Industrie & services,-1.16,1.20,0.39,-0.65,-1.08,1.11,-0.41
Société de financement,-0.21,0.21,0.13,-0.18,-0.29,0.30,-0.10
Établissement public,-1.35,1.41,0.22,-0.53,-1.02,1.05,-0.31


In [9]:
scenarios.groupby("nature_taux")[list(SCENARIOS)].mean().round(2)

,Hausse parallèle,Baisse parallèle,Pentification,Aplatissement,Hausse des taux courts,Baisse des taux courts,"BAM +50 pb, 10 ans -20 pb"
nature_taux,,,,,,,
fixe,-2.73,2.89,0.07,-0.68,-1.55,1.59,-0.36
révisable,-0.39,0.40,0.22,-0.32,-0.52,0.53,-0.20


### Choc de spread

La prime de risque ne se révise jamais : une tranche révisable est protégée contre les taux de BAM, **pas** contre une hausse de la prime de son émetteur. Exemple : +100 pb sur l'immobilier, +50 pb sur les sociétés de financement.

In [10]:
spread = impact_scenario(resultats, courbe, chocs_pb=None,
                         spreads_pb={"Immobilier": 100, "Société de financement": 50})
spread.pivot_table(index="secteur", columns="nature_taux", values="variation_%", aggfunc="mean").round(2)

nature_taux,fixe,révisable
secteur,,
Banque,0.00,0.00
Immobilier,-1.42,-1.82
Industrie & services,0.00,0.00
Société de financement,NaN,-1.88
Établissement public,0.00,0.00


## 5. Exports

In [11]:
print(exporter_excel(resultats, courbe))
print(exporter_html(resultats, courbe))

data/rapport_resultats.xlsx
data/rapport.html


## Conclusions

*À compléter avec tes chiffres :*

- **Structure du marché** : … % des tranches vivantes sont à taux révisable ; les banques représentent … tranches sur ….
- **Risque de taux** : duration de taux moyenne de … ; le risque se concentre sur … (secteur, maturité).
- **Risque de crédit** : prime moyenne de … pb (banques) à … pb (immobilier).
- **Scénarios** : le scénario le plus pénalisant pour le marché est … ; les tranches fixes de moyen terme …
- **Spread** : les révisables ont une duration de taux d'environ … mais une duration de spread de …

In [12]:
resultats[resultats["differe_ans"] > 0][["emetteur", "tranche", "differe_ans", "maturite_res", "modifiee"]]

,emetteur,tranche,differe_ans,maturite_res,modifiee
30,ALLIANCES DÉVELOPPEMENT IMMOBILIER,A,3,6.12,0.109
37,ATTIJARIWAFA BANK,A,2,5.12,0.117
40,ATTIJARIWAFA BANK,B,2,5.12,0.117
47,BCP,A,5,7.13,0.125
66,ALLIANCES DARNA,A,3,3.22,0.213
77,ATTIJARIWAFA BANK,B,2,4.63,0.608
78,ATTIJARIWAFA BANK,A,2,4.63,0.608
87,ATTIJARIWAFA BANK,A,2,5.64,0.613
88,ATTIJARIWAFA BANK,B,2,5.64,0.613
107,BMCI,A,5,7.87,0.840
